In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

failures = []

# CHECK I: row count + freshness per layer

# row count — asnje tabele bosh
for layer, table in [("bronze", "orders"), ("silver", "orders"), ("gold", "fact_orders")]:
    cnt = spark.table(f"{catalog_name}.{layer}.{table}").count()
    if cnt == 0:
        failures.append(f"{layer}.{table} is empty")
    print(f"{layer}.{table}: {cnt:,} rows")

# freshness — data me e fundit e porosive
max_order_date = spark.table(f"{catalog_name}.{silver_schema}.orders") \
    .agg(F.max("order_purchase_timestamp")).collect()[0][0]
print(f"Latest order date: {max_order_date}")
if max_order_date is None:
    failures.append("orders has no dates")
elif max_order_date.year < 2018:
    failures.append(f"orders data looks stale — latest is {max_order_date}")

# CHECK II: uniqueness of dimension keys
dim_keys = {
    "dim_customer": "customer_unique_id",
    "dim_product": "product_id",
    "dim_seller": "seller_id",
    "dim_date": "date_key",
    "dim_geography": "zip_code_prefix",
}
for dim, key in dim_keys.items():
    df = spark.table(f"{catalog_name}.{gold_schema}.{dim}")
    total = df.count()
    distinct = df.select(key).distinct().count()
    if total != distinct:
        failures.append(f"{dim}.{key} not unique: {total} rows, {distinct} distinct")
    print(f"{dim}.{key}: {'unique' if total==distinct else 'DUPLICATE'}")

In [0]:
# CHECK III: referential integrity facts => dims (te gjitha)
ref_checks = [
    ("fact_orders",      "customer_unique_id", "dim_customer", "customer_unique_id"),
    ("fact_orders",      "date_key",           "dim_date",     "date_key"),
    ("fact_order_items", "customer_unique_id", "dim_customer", "customer_unique_id"),
    ("fact_order_items", "product_id",         "dim_product",  "product_id"),
    ("fact_order_items", "seller_id",          "dim_seller",   "seller_id"),
    ("fact_order_items", "date_key",           "dim_date",     "date_key"),
    ("fact_payments",    "date_key",           "dim_date",     "date_key"),
]

for fact, fk, dim, pk in ref_checks:
    f = spark.table(f"{catalog_name}.{gold_schema}.{fact}").select(fk).where(F.col(fk).isNotNull())
    d = spark.table(f"{catalog_name}.{gold_schema}.{dim}").select(F.col(pk).alias(fk))
    orphans = f.join(d, on=fk, how="left_anti").count()
    if orphans > 0:
        failures.append(f"{fact}.{fk} has {orphans} orphans not in {dim}")
    print(f"{fact}.{fk} -> {dim}.{pk}: {orphans} orphans")

# CHECK IV: range on price/freight/review_score
items = spark.table(f"{catalog_name}.{gold_schema}.fact_order_items")
fact = spark.table(f"{catalog_name}.{gold_schema}.fact_orders")

bad_price = items.filter((F.col("price") <= 0)).count()
bad_freight = items.filter((F.col("freight_value") < 0)).count()
bad_score = fact.filter(
    (F.col("review_score") < 1) | (F.col("review_score") > 5)
).filter(F.col("review_score").isNotNull()).count()

if bad_price > 0: failures.append(f"{bad_price} items with price <= 0")
if bad_freight > 0: failures.append(f"{bad_freight} items with freight < 0")
if bad_score > 0: failures.append(f"{bad_score} reviews with score out of 1-5")
print(f"Range: price<=0:{bad_price}, freight<0:{bad_freight}, score out:{bad_score}")

In [0]:
# GATE: nese ndonje check deshtoi, NDAL job-in
if failures:
    print("\n=== DATA QUALITY FAILURES ===")
    for f in failures:
        print(f"  ✗ {f}")
    raise Exception(f"Data quality gate failed: {len(failures)} issue(s)")
else:
    print("\n✓ All data quality checks passed")